# 01 — The Iterator Protocol

Every `for x in something:` in Python desugars to:

```python
it = iter(something)              # __iter__
while True:
    try:
        x = next(it)              # __next__
    except StopIteration:
        break
    # ... loop body ...
```

Two things: **iterable** (something you can ask for an iterator) and **iterator** (the thing that produces one value at a time).

## `iter()` and `next()` by hand

In [1]:
xs = [10, 20, 30]            # `xs` is an iterable (list)
it = iter(xs)                # `it` is the iterator

print(type(xs).__name__, type(it).__name__)
print(next(it))              # 10
print(next(it))              # 20
print(next(it))              # 30

try:
    next(it)
except StopIteration:
    print("exhausted")

list list_iterator
10
20
30
exhausted


### Iterators are one-shot

Once an iterator is exhausted, it's done. You'd need a *fresh* iterator (`iter(xs)` again) to start over. This is also why generator expressions are single-use — they ARE iterators.

In [2]:
it = iter([1, 2, 3])
print(list(it))    # [1, 2, 3]
print(list(it))    # []  — same iterator, already exhausted

# But the underlying list is fine — you can re-iterate it as many times as you want:
xs = [1, 2, 3]
print(list(iter(xs)), list(iter(xs)))

[1, 2, 3]
[]
[1, 2, 3] [1, 2, 3]


## `next(it, default)` — no exception needed

The 2-arg form returns the default when the iterator is exhausted instead of raising `StopIteration`. Handy in pipelines.

In [3]:
it = iter([])
print(next(it, None))         # None — never raises

# Common pattern: "first element matching X, or None".
users = [{"id": 1, "role": "admin"}, {"id": 2, "role": "user"}, {"id": 3, "role": "admin"}]
first_admin = next((u for u in users if u["role"] == "admin"), None)
print(first_admin)

None
{'id': 1, 'role': 'admin'}


## Writing a custom iterator class

Two dunders: `__iter__` returns *the iterator* (often `self`), and `__next__` returns *the next value* (or raises `StopIteration`).

In [4]:
class CountUp:
    """A simple iterator: yields 0, 1, ..., stop-1."""

    def __init__(self, stop: int):
        self.stop = stop
        self._i = 0

    def __iter__(self):
        # An iterator is its own iterator. (An *iterable* might return a NEW iterator each time.)
        return self

    def __next__(self):
        if self._i >= self.stop:
            raise StopIteration
        v = self._i
        self._i += 1
        return v

for x in CountUp(4):
    print(x)

# We can also use it manually:
c = CountUp(2)
print(next(c), next(c), next(c, "done"))

0
1
2
3
0 1 done


### Iterable vs iterator — the difference matters

If you want your container to be **re-iterable** (like `list`), `__iter__` should return a *new* iterator each time. The container itself is *not* the iterator.

In [5]:
class Cards:
    """An iterable that hands out a fresh iterator each time."""

    def __init__(self, items):
        self._items = list(items)

    def __iter__(self):
        # A new iterator every call — so two `for`s over the same Cards both work.
        return iter(self._items)

deck = Cards(["A", "K", "Q"])
for c in deck: print(c)
print("---")
for c in deck: print(c)    # still works

A
K
Q
---
A
K
Q


## The world is full of iterables

Almost everything in stdlib that returns a sequence is iterable: `range`, `dict.items()`, `enumerate`, `zip`, `map`, `filter`, `open(...)` (yields lines), `os.walk`. You rarely need to write `__iter__` by hand — generators (next notebook) take care of 95% of cases.

In [6]:
# `range` doesn't build a list — it's lazy.
r = range(1_000_000_000)
print(r)                 # range object, not a giant list
print(next(iter(r)))     # 0

# `map` and `filter` are lazy too — they return iterators, not lists.
m = map(str.upper, ["a", "b", "c"])
print(type(m).__name__)
print(list(m))
print(list(m))           # exhausted — empty

range(0, 1000000000)
0
map
['A', 'B', 'C']
[]


## Mini-exercise

1. Write a `Fibonacci(n)` iterator class that yields the first `n` Fibonacci numbers.
2. Make `Fibonacci` *re-iterable* (each call to `iter(fib)` starts from 0 again). Hint: separate the **iterable** class from a small **iterator** inner class.
3. What does this print? Why?
   ```python
   xs = [1, 2, 3]
   it1 = iter(xs); it2 = iter(xs)
   print(next(it1), next(it2), next(it1), next(it2))
   ```

In [7]:
# your solutions here


**Takeaways**

- `iter(x)` returns an iterator; `next(it)` pulls the next value; `StopIteration` ends iteration.
- Iterators are **one-shot**. Iterables can hand out fresh iterators on demand.
- `next(it, default)` is the bare-handed way to avoid the `StopIteration` dance.
- You can write `__iter__` / `__next__` by hand, but generators (next notebook) usually win.

Next: [02_generators.ipynb](02_generators.ipynb) — `yield` the easy way.